# NB05 — Reading Single-Cell and Transcriptomics Papers

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2.5 hours  

---

## Motivation

Single-cell RNA sequencing (scRNA-seq) has become one of the dominant methodological frameworks in modern biology. A computational biology candidate who cannot read a scRNA-seq paper fluently — identifying the analysis choices, evaluating the claims, and spotting the limitations — is missing literacy in a field that now appears in the methods section of papers across oncology, developmental biology, immunology, and neuroscience.

This notebook builds that literacy without requiring access to real scRNA-seq data.

## 1. What to Look for in a scRNA-seq Paper

### In the methods section:

| Step | What to find | Common omissions |
|------|-------------|------------------|
| Library preparation | 10x Genomics version, chemistry, read depth | Chemistry version omitted (affects UMI counts) |
| Quality filtering | Min UMI, min genes, max mitochondrial % | Thresholds not stated — "we filtered low-quality cells" |
| Normalization | Method (library-size, scran, Pearson residuals), scale factor | Method not stated — "we normalized" |
| Feature selection | Number of highly variable genes, method | Feature count not stated |
| Dimensionality reduction | PCA components used before UMAP/t-SNE | Components not stated |
| Clustering | Algorithm (Leiden/Louvain), resolution parameter | Resolution omitted — cluster number is resolution-dependent |
| Marker genes | Method (Wilcoxon, logistic regression), adjusted p-value threshold | Test not stated |
| Integration | Harmony/Seurat CCA — why it was used | Integration not justified |

### Key papers to know:

- **Seurat v2** (Butler et al. 2018 Nature Biotech) — Canonical Correlation Analysis for integration
- **Scanpy** (Wolf et al. 2018 Genome Biology) — Python ecosystem for scRNA-seq
- **Harmony** (Korsunsky et al. 2019 Nature Methods) — fast, flexible integration

## 2. Reading a UMAP: What You Can and Cannot Conclude

### What UMAP shows:
- Relative proximity of cells in high-dimensional gene expression space (approximately)
- Cluster separation (cells of similar type tend to group together)
- Potential batch effects (if two experimental batches separate without biological reason)

### What UMAP does NOT show:
- True distances between clusters (UMAP distorts global distances; only local structure is reliable)
- Whether two clusters are truly distinct cell types or the same type at different states
- Absolute gene expression values
- Statistical significance of any separation

**Critical warning:** UMAP will always produce clusters even from random data. The biological meaning of clusters must be established by marker gene analysis and external validation — not by how separated they look on the UMAP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import random as sparse_random
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

np.random.seed(42)


def simulate_scrna_counts(n_cells: int = 500, n_genes: int = 2000,
                           n_cell_types: int = 5) -> tuple:
    """Simulate a synthetic scRNA-seq count matrix with known cell type structure.

    Returns: (counts matrix [n_cells x n_genes], cell_type_labels [n_cells])
    """
    cells_per_type = n_cells // n_cell_types
    counts_list = []
    labels = []

    # Each cell type has a characteristic gene expression program
    for ct in range(n_cell_types):
        # Signature genes: 100 genes expressed highly in this cell type
        signature = np.zeros(n_genes)
        sig_genes = np.random.choice(n_genes, 100, replace=False)
        signature[sig_genes] = np.random.uniform(3, 8, 100)

        # Generate counts: Poisson with cell-type-specific mean
        base_expression = np.random.uniform(0, 0.5, n_genes)  # background
        mean_expr = base_expression + signature

        for _ in range(cells_per_type):
            # Vary total UMI count per cell (library size variation)
            cell_depth = np.random.randint(2000, 8000)
            probs = np.exp(mean_expr + np.random.normal(0, 0.2, n_genes))
            probs = probs / probs.sum()
            cell_counts = np.random.multinomial(cell_depth, probs)
            counts_list.append(cell_counts)
            labels.append(f'CellType_{ct+1}')

    return np.array(counts_list), np.array(labels)


def pipeline_filter(counts: np.ndarray,
                    min_umis: int = 500, min_genes: int = 200,
                    max_mt_fraction: float = 0.20) -> np.ndarray:
    """Quality filter cells.

    Simulates mitochondrial gene fraction (last 10% of genes = 'MT genes').
    Returns boolean mask of cells passing QC.
    """
    total_umi = counts.sum(axis=1)
    n_genes_expressed = (counts > 0).sum(axis=1)
    mt_genes = counts[:, -int(counts.shape[1] * 0.1):]
    mt_fraction = mt_genes.sum(axis=1) / (total_umi + 1)

    mask = (total_umi >= min_umis) & (n_genes_expressed >= min_genes) & (mt_fraction <= max_mt_fraction)
    return mask


def pipeline_normalize(counts: np.ndarray, scale_factor: float = 1e4) -> np.ndarray:
    """Library-size normalization (CPM) + log1p transform."""
    total = counts.sum(axis=1, keepdims=True)
    normalized = counts / total * scale_factor
    return np.log1p(normalized)


def select_hvg(log_counts: np.ndarray, n_top: int = 500) -> np.ndarray:
    """Select highly variable genes by variance-to-mean ratio.

    Returns indices of top n_top most variable genes.
    """
    gene_means = log_counts.mean(axis=0)
    gene_vars = log_counts.var(axis=0)
    dispersion = gene_vars / (gene_means + 1e-8)
    return np.argsort(dispersion)[::-1][:n_top]


def simple_umap_embed(pca_coords: np.ndarray, n_components: int = 2) -> np.ndarray:
    """Simplified 2D embedding using PCA (approximates UMAP structure for synthetic data).

    Real analysis would use the umap-learn library.
    For synthetic data with well-separated cell types, PCA 2D approximates UMAP structure.
    """
    return pca_coords[:, :n_components]


# Run the synthetic pipeline
print("Simulating scRNA-seq data...")
counts, labels = simulate_scrna_counts(n_cells=500, n_genes=2000)
print(f"Raw matrix: {counts.shape[0]} cells x {counts.shape[1]} genes")
print(f"Total UMI range: {counts.sum(axis=1).min()}–{counts.sum(axis=1).max()}")

qc_mask = pipeline_filter(counts)
counts_filt = counts[qc_mask]
labels_filt = labels[qc_mask]
print(f"After QC filter: {counts_filt.shape[0]} cells ({qc_mask.sum()}/{len(qc_mask)} passed)")

log_counts = pipeline_normalize(counts_filt)
hvg_idx = select_hvg(log_counts, n_top=500)
log_hvg = log_counts[:, hvg_idx]
print(f"After HVG selection: {log_hvg.shape[1]} genes")

pca = PCA(n_components=30, random_state=42)
pca_coords = pca.fit_transform(log_hvg)
embed_2d = simple_umap_embed(pca_coords)
print(f"PCA embedding: {pca_coords.shape}, 2D for visualization: {embed_2d.shape}")

In [ ]:
# Marker gene analysis: top genes per cell type
from scipy import stats as scipy_stats

unique_types = np.unique(labels_filt)
cell_type_colors = plt.cm.Set1(np.linspace(0, 1, len(unique_types)))
color_map = dict(zip(unique_types, cell_type_colors))

# Find top 5 marker genes per cell type (Wilcoxon-like: high vs rest)
top_markers_per_type = {}
for ct in unique_types:
    ct_mask = labels_filt == ct
    ct_expr = log_hvg[ct_mask].mean(axis=0)
    other_expr = log_hvg[~ct_mask].mean(axis=0)
    log_fc = ct_expr - other_expr
    top_idx = np.argsort(log_fc)[::-1][:5]
    top_markers_per_type[ct] = top_idx

# Build dotplot data: for top 3 markers per type (15 genes total)
marker_genes = []
for ct in unique_types:
    marker_genes.extend(top_markers_per_type[ct][:3])
marker_genes = list(dict.fromkeys(marker_genes))  # deduplicate preserving order

# Compute mean expression and fraction expressed for dotplot
dot_means = np.zeros((len(unique_types), len(marker_genes)))
dot_fracs = np.zeros((len(unique_types), len(marker_genes)))
for i, ct in enumerate(unique_types):
    ct_mask = labels_filt == ct
    ct_expr = log_hvg[ct_mask][:, marker_genes]
    dot_means[i] = ct_expr.mean(axis=0)
    dot_fracs[i] = (ct_expr > 0).mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Synthetic scRNA-seq Analysis Pipeline', fontsize=13, fontweight='bold')

# Panel 1: UMAP (PCA 2D)
ax1 = axes[0]
for ct, color in zip(unique_types, cell_type_colors):
    mask = labels_filt == ct
    ax1.scatter(embed_2d[mask, 0], embed_2d[mask, 1],
                c=[color], s=10, alpha=0.7, label=ct)
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('PCA Embedding\n(approximates UMAP structure)')
ax1.legend(markerscale=2, fontsize=8, loc='upper right')
ax1.grid(True, alpha=0.2)

# Panel 2: Marker gene dotplot
ax2 = axes[1]
xx, yy = np.meshgrid(range(len(marker_genes)), range(len(unique_types)))
scatter = ax2.scatter(xx.ravel(), yy.ravel(),
                       s=dot_fracs.ravel() * 200,
                       c=dot_means.ravel(), cmap='Reds',
                       vmin=0, vmax=dot_means.max())
plt.colorbar(scatter, ax=ax2, label='Mean log-normalized expression')
ax2.set_yticks(range(len(unique_types)))
ax2.set_yticklabels(unique_types, fontsize=9)
ax2.set_xticks(range(len(marker_genes)))
ax2.set_xticklabels([f'Gene_{g}' for g in marker_genes], rotation=90, fontsize=7)
ax2.set_title('Marker Gene Dotplot\n(size = fraction expressed)')
ax2.grid(True, alpha=0.2)

# Panel 3: Pipeline flowchart (text-based)
ax3 = axes[2]
ax3.axis('off')
steps = [
    ('Raw counts\n(500 cells, 2000 genes)', '#E3F2FD'),
    ('QC Filter\n(min UMI, min genes,\nmax MT%)', '#FFF3E0'),
    ('Normalize\n(CPM + log1p)', '#E8F5E9'),
    ('Select HVGs\n(top 500 by\ndispersion)', '#F3E5F5'),
    ('PCA\n(30 components)', '#FCE4EC'),
    ('2D Embedding\n(PCA/UMAP)', '#E0F2F1'),
    ('Cluster\n(Leiden/Louvain)', '#FFF8E1'),
    ('Marker genes\n(Wilcoxon test)', '#E8EAF6'),
]
y_positions = np.linspace(0.95, 0.05, len(steps))
for (label, color), y in zip(steps, y_positions):
    ax3.text(0.5, y, label, ha='center', va='center', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', facecolor=color, edgecolor='gray', linewidth=0.8),
             transform=ax3.transAxes)
    if y > y_positions[-1] + 0.01:
        next_y_idx = list(y_positions).index(y) + 1
        if next_y_idx < len(y_positions):
            ax3.annotate('', xy=(0.5, y_positions[next_y_idx] + 0.04),
                         xytext=(0.5, y - 0.04),
                         xycoords='axes fraction', textcoords='axes fraction',
                         arrowprops=dict(arrowstyle='->', color='gray'))
ax3.set_title('Analysis Pipeline Flowchart', fontsize=10)

plt.tight_layout()
plt.savefig('scrna_paper_reading.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Read the abstract and figures of Wolf et al. (2018) SCANPY. Apply Pass 1. What is the one-sentence summary?

**Exercise 2:** Find any scRNA-seq paper (from Module 10's papers.md or a recent preprint). Audit its methods section using the table in Section 1 of this notebook. What is missing?

**Exercise 3:** Why is the Leiden clustering resolution parameter important to report? What happens to the number of clusters as resolution increases?

**Exercise 4 (reflection):** In one sentence: why does the UMAP looking visually separated not constitute evidence that the clusters are biologically distinct cell types?

## Quiz

1. Name three pieces of information that must appear in a scRNA-seq methods section but are often missing.
2. What does mitochondrial gene fraction indicate in a single-cell experiment?
3. What is the difference between Seurat integration and Harmony integration?
4. Name two things a UMAP shows and two things it does not show.
5. What statistical test is commonly used to find marker genes in Seurat/Scanpy?

## References

- Butler et al. (2018). Integrating single-cell transcriptomic data. Nature Biotech 36:411–420.
- Wolf et al. (2018). SCANPY. Genome Biology 19:15.
- Korsunsky et al. (2019). Harmony. Nature Methods 16:1289–1296.
- Luecken & Theis (2019). Current best practices in single-cell RNA-seq analysis. Mol Syst Biol 15:e8746.